# DTU Social Data (2026) - Final Project Explainer
**Group Members:** Andrea De Pascale, [Groupmate 1 Name], [Groupmate 2 Name]

---

## 1. Motivation

**What is your dataset?**
Our analysis uses three datasets:
1. **HFUDD16** (Statistics Denmark): Data on education levels and socioeconomic status (employed, unemployed, enrolled) per municipality.
2. **IFOR35** (Statistics Denmark): Historical average income data per municipality.
3. **Company Data**: Locations and industries of major companies in Denmark.

**Why did you choose these particular datasets?**
Combining these datasets allows us to connect a municipality's wealth with its demographics and corporate presence. Instead of just mapping income levels, linking HFUDD16 and IFOR35 lets us analyze the makeup of the local workforce. Adding the company data helps identify if specific industries drive municipal wealth.

**What was your goal for the end user's experience?**
We aimed for a "Martini Glass" structure. We want to present our main findings first (national inflation trends, demographic uniformity, and specific regional anomalies) as the "stem". Then, we open the "glass" by providing an interactive dashboard where users can filter the data and explore individual municipalities themselves.

In [1]:
# Import necessary libraries for the analysis
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Suppress warnings for cleaner notebook presentation
import warnings
warnings.filterwarnings('ignore')

## 2. Basic stats

### Data Cleaning and Preprocessing
To create the interactive visualizations, we preprocessed the data as follows:
* **String Normalization:** We standardized municipality names across datasets by removing suffixes (e.g., "Kommune") and standardizing Danish characters (æ, ø, å) to ensure reliable merges.
* **Merging Data:** We merged the `HFUDD16` and `IFOR35` datasets using an inner join on the `Municipality` and `Year` columns.
* **Ratio Calculations:** For the ternary plot, we converted raw population counts into ratios (Employed, Unemployed, Enrolled) per municipality to accurately different regions.

*(Groupmates: Add your data cleaning steps for the company and education datasets here).*

### Exploratory Data Analysis
During this exploratory analysis, we plotted the national average income from 2008 to 2024. The data shows a steady, linear increase across the country, which scales as expected with national inflation and wage adjustments.

In [2]:
# Load your merged dataset
df = pd.read_csv('src/Part1/data/Part1_merged_dataset.csv')

# Display the first few rows and basic statistical summary
display(df.head())

# ---------------------------------------------------------
# Plotting the National Inflation Trend
# ---------------------------------------------------------
# Group by year and calculate the mean average income across ALL municipalities
# This creates our baseline "National Trend"
national_trend = df.groupby('Year')['Average Income'].mean().reset_index()

# Filter for the years 2008 to 2024 as mentioned in the text
national_trend = national_trend[(national_trend['Year'] >= 2008) & (national_trend['Year'] <= 2024)]

# Create the line chart
fig_trend = px.line(
    national_trend, 
    x='Year', 
    y='Average Income',
    title="National Average Income Trend (2008-2024)",
    labels={'Average Income': 'Average Income (DKK)', 'Year': 'Year'},
    markers=True
)

# Format the y-axis to include commas for easier reading of large numbers
fig_trend.update_layout(yaxis_tickformat=',')
fig_trend.show()

,Municipality,Year,Average Income,Employed_Ratio,Unemployed_Ratio,Enrolled_Ratio,Outside_Ratio
0,Aabenraa,2008,257029.5,0.587436,0.016870,0.110913,0.284782
1,Aabenraa,2009,255130.2,0.558074,0.034676,0.114738,0.292512
2,Aabenraa,2010,271481.9,0.547915,0.034071,0.121560,0.296454
3,Aabenraa,2011,268613.8,0.536807,0.035979,0.123741,0.303473
4,Aabenraa,2012,268157.0,0.529620,0.039931,0.124723,0.305727


## 3. Data Analysis

### Part 1: Geographical distribution & Socioeconomic anomalies
Analyzing the merged HFUDD16 and IFOR35 datasets revealed two main points:
1. **Demographic uniformity:** The ratios of employed, unemployed, and enrolled individuals are highly similar across all municipalities. Since the workforce composition doesn't change much geographically, regional income differences are likely driven by the types of industries present rather than workforce size or employment rates.
2. **Average income steady increase:** The average income mostly increases gradually for all municipalities from 2008 to 2024, likely due to the inflation rates of the last decades.
3. **The Vejen & Billund anomalies:** In 2023, average incomes spiked in Vejen (from 314k to 446k DKK) and Billund (from 329k to 470k DKK). In 2024, Billund's average income dropped back to 340k DKK, while Vejen's stayed high at 435k DKK.

### Part 2: Industry vs. Income ([Groupmate 1 Name])
*(Groupmate 1: Briefly state your findings linking company locations to municipal wealth).*

### Part 3: Education to Industry Pipeline ([Groupmate 2 Name])
*(Groupmate 2: Briefly state your findings on how education levels map to specific corporate sectors).*

In [8]:
# ---------------------------------------------------------
# Plotting the Vejen & Billund Anomaly vs. National Baseline
# ---------------------------------------------------------

# 1. Filter for recent years to clearly see the lead-up to the 2023 spike
df_recent = df[df['Year'] >= 2008].copy()

# 2. Create a new category column. 
# If the municipality is Vejen or Billund, it keeps its name. 
# Otherwise, it gets grouped into 'Rest of Denmark'
df_recent['Category'] = df_recent['Municipality'].apply(
    lambda x: x if x in ['Vejen', 'Billund'] else 'Rest of Denmark Average'
)

# 3. Group by Year and the new Category to calculate the mean income
df_anomaly_comparison = df_recent.groupby(['Year', 'Category'])['Average Income'].mean().reset_index()

# 4. Define specific colors so the baseline fades into the background
# and the anomalies stand out brightly.
custom_colors = {
    'Vejen': '#e74c3c',                 # Red
    'Billund': '#3498db',               # Blue
    'Rest of Denmark Average': '#bdc3c7' # Neutral Gray
}

# 5. Create the line chart
fig_anomaly = px.line(
    df_anomaly_comparison, 
    x='Year', 
    y='Average Income', 
    color='Category', 
    color_discrete_map=custom_colors,
    markers=True,
    title="The 2023 income spike vs. the rest of Denmark",
    labels={'Average Income': 'Average Income (DKK)', 'Year': 'Year', 'Category': 'Region'}
)

# 6. Format the y-axis and make the gray line thicker/dashed if desired
fig_anomaly.update_layout(
    yaxis_tickformat=',',
    plot_bgcolor='rgba(240,240,240,0.5)' # Slight background to make lines pop
)

# Optional: Make the baseline line dashed to separate it further from the actual municipalities
fig_anomaly.update_traces(
    patch={"line": {"dash": "dash"}}, 
    selector={"legendgroup": "Rest of Denmark Average"}
)

fig_anomaly.show()

## 4. Genre

We used the **Partitioned Poster** genre combined with a **Martini Glass** narrative structure.

### Visual Narrative (Tools from Segel & Heer)
* **Consistent Visual Platform:** We built all visualizations using Plotly so the design and interaction logic remain standardized across the page.
* **Feature Distinction & Close-ups:** In the dashboard, clicking a municipality highlights it on the map and isolates its specific data point on the ternary plot, acting as a close-up.
* **Familiar Objects:** We use a map of Denmark as the primary navigation tool, giving users an intuitive way to filter complex demographic data.

### Narrative Structure (Tools from Segel & Heer)
* **Linear Ordering:** The top sections of the webpage dictate the reading order. We guide the user through national trends, industry impact, and education flows.
* **Filtering / Selection:** At the bottom of each section, we provide interactive tools (Dash and Plotly elements) so the user can filter the data themselves.
* **Multi-messaging:** Each chart is paired with dedicated text blocks that explicitly state the main takeaway before the user interacts with the visualization.

## 5. Visualizations

### The Interactive Map & Ternary Plot
* **Choropleth Map:** This shows the spatial distribution of wealth, making it easy to identify high-income and low-income regions.
* **Ternary Plot:** Socioeconomic status (Employed, Unemployed, Enrolled) is compositional data. A ternary plot is the standard method for visualizing three variables that represent parts of a whole.
* **Why they fit the story:** Cross-filtering these plots proves the concept of demographic uniformity. When users click on municipalities with drastically different income levels on the map, the highlighted point on the ternary plot barely moves.

### Industry Distribution Plot ([Groupmate 1 Name])
*(Groupmate 1: State what plot you used and exactly why it fits your specific data).*

### Education Flow Sankey Diagram ([Groupmate 2 Name])
*(Groupmate 2: State why a Sankey diagram is the best choice for showing the movement from education to industry).*

## 6. Discussion

**What went well?**
We successfully merged spatial, financial, and educational data. Deploying a live Python backend on Render.com and embedding it into a static GitHub Pages site allowed us to achieve two-way cross-filtering (linking the map to the ternary plot), which isn't natively supported by static HTML exports.

**What could be improved?**
Our data highlights anomalies but lacks the granularity to explain them. For example, we cannot determine if the 2023 income spikes in Vejen and Billund were caused by large corporate bonuses or a sudden shift in taxpayer demographics. Answering this requires historical data on individual migrations or company financial reports. Additionally, relying only on "main" companies ignores the economic impact of small and medium enterprises on municipal wealth.

## 7. Contributions

While all group members participated in conceptualizing the data story and formatting the final GitHub webpage, the primary technical responsibilities were divided as follows:

* **Andrea De Pascale (s243094):** Led Part 1. Responsible for the loading via API calls and merging of the HFUDD16 and IFOR35 datasets, normalizing municipality GeoJSON data, and building the interactive Dash application. Managed the cloud deployment on Render to enable two-way cross-filtering between the Choropleth map and Ternary plot.
* **[Groupmate 1 Name]:** Led Part 2. Responsible for cleaning and integrating the company location dataset, conducting the exploratory analysis on corporate impact, and generating the static interactive visualizations relating industry to municipal wealth.
* **[Groupmate 2 Name]:** Led Part 3. Responsible for mapping the pathways from education levels to industry sectors. Designed and coded the complex Sankey diagrams to visualize the flow of the workforce.

## 8. References

1. **Statistics Denmark (Danmarks Statistik):**
   * *HFUDD16 Dataset*: Population by education, socioeconomic status, and municipality.
   * *IFOR35 Dataset*: Average income by municipality and deciles.
2. **Segel, E., & Heer, J. (2010):**
   * *Narrative Visualization: Telling Stories with Data*. IEEE Transactions on Visualization and Computer Graphics. (Used for genre and structural framework).
3. **Click That Hood (GitHub):** Open-source repository used for the GeoJSON borders of Danish municipalities.